In [3]:
import numpy as np
import jax.numpy as jnp

# Cargar los datos
datos = np.load("datos_3430.npz")

# Extraer las señales
X = jnp.array(datos['X'])

y = jnp.array(datos['y'])

print(f"Forma de los datos X: {X.shape}")
print(f"Forma de las etiquetas y: {y.shape}")

Forma de los datos X: (36, 256)
Forma de las etiquetas y: (36,)


In [8]:
def inicializar_aleatorio(key):
    key1, key2 = random.split(key)
    # Inicialización aleatoria estándar escalada para evitar saturación de la tanh
    W1 = random.normal(key1, (30, 256)) * jnp.sqrt(1.0 / 256.0)
    W2 = random.normal(key2, (256, 30)) * jnp.sqrt(1.0 / 30.0)
    return [W1, W2]

def inicializar_fourier(key):
    # Crear vector de tiempo normalizado t
    t = jnp.linspace(0, 1, 256, endpoint=False)

    # Crear vector de frecuencias k = 1, ..., 15
    k = jnp.arange(1, 16)

    # Vectorización: Expandir dimensiones para producto externo (15, 256)
    t_mat = t[None, :]  # Forma (1, 256)
    k_mat = k[:, None]  # Forma (15, 1)

    # Calcular senos y cosenos (Forma: 15x256 cada uno)
    senos = jnp.sin(2 * jnp.pi * k_mat * t_mat)
    cosenos = jnp.cos(2 * jnp.pi * k_mat * t_mat)

    # Apilar senos y cosenos para formar W1 (Forma: 30x256)
    W1 = jnp.vstack([senos, cosenos])

    # W2 se inicializa aleatoriamente en ambos casos
    _, key2 = random.split(key)
    W2 = random.normal(key2, (256, 30)) * jnp.sqrt(1.0 / 30.0)

    return [W1, W2]

In [9]:
@jit
def forward(params, x):
    W1, W2 = params
    # Asumiendo que 'x' entra como un batch de forma (N, 256)
    # W1.T tiene forma (256, 30), el resultado de x @ W1.T es (N, 30)
    z1 = jnp.dot(x, W1.T)
    a1 = jnp.tanh(z1)
    # W2.T tiene forma (30, 256), el resultado de a1 @ W2.T es (N, 256)
    x_hat = jnp.dot(a1, W2.T)
    return x_hat

@jit
def loss_fn(params, x):
    x_hat = forward(params, x)
    # Media cuadrática del error
    return jnp.mean((x_hat - x)**2)

@jit
def update(params, x, lr):
    # Calcula el valor de la pérdida y los gradientes respecto a 'params'
    loss, grads = jax.value_and_grad(loss_fn)(params, x)

    # Actualización de Descenso de Gradiente
    new_params = [p - lr * g for p, g in zip(params, grads)]
    return new_params, loss

In [ ]:
def entrenar_red(params_iniciales, datos_x, iteraciones=4000, lr=0.02):
    params = params_iniciales
    historial_loss = []

    for i in range(iteraciones):
        params, loss = update(params, datos_x, lr)
        historial_loss.append(loss)

        if (i + 1) % 500 == 0:
            print(f"Iteración {i+1}/{iteraciones} | Loss: {loss:.6f}")

    return params, historial_loss

# Generamos las semillas para la aleatoriedad
key = random.PRNGKey(42)

# 1. Entrenar con inicialización Aleatoria
print("--- Entrenando con Inicialización Aleatoria ---")
params_rand = inicializar_aleatorio(key)
params_rand_final, loss_rand = entrenar_red(params_rand, X)

# 2. Entrenar con inicialización de Fourier
print("\n--- Entrenando con Inicialización de Fourier ---")
params_fourier = inicializar_fourier(key)
params_fourier_final, loss_fourier = entrenar_red(params_fourier, X)